In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('drug.csv')
data

In [ ]:
class_att = data.columns[-1]
class_att

In [ ]:
classes = data[class_att].unique()
classes

In [ ]:
apriori = data[class_att].value_counts()
apriori

In [ ]:
apriori = apriori / apriori.sum()
apriori

In [ ]:
log_apriori = np.log(apriori)
log_apriori

In [ ]:
features = data.drop(columns=[class_att])
features.columns

In [ ]:
col = 'Age'
pd.api.types.is_numeric_dtype(data[col])

In [ ]:
stats = data.groupby(class_att)[col].agg(['mean', 'std'])
stats

In [ ]:
stats['std'] = stats['std'].replace(0, 1e-9)
stats

In [ ]:
col = 'Sex'
pd.api.types.is_numeric_dtype(data[col])

In [ ]:
mat = pd.crosstab(data[col], data[class_att])
mat

In [ ]:
n_values = mat.shape[0]
n_values

In [ ]:
smoothing = 1.0
mat_smooth = (mat + smoothing) / (mat.sum(axis=0) + smoothing * n_values)
mat_smooth

In [ ]:
np.log(mat_smooth)

In [ ]:
def is_numeric(data, col):
    return pd.api.types.is_numeric_dtype(data[col])

In [ ]:
def learn(data, class_att, smoothing=1.0):
    model = {}
    classes = data[class_att].unique()
    model['_classes'] = classes

    apriori = data[class_att].value_counts()
    apriori = apriori / apriori.sum()
    model['_apriori'] = np.log(apriori)

    for col in data.drop(columns=[class_att]).columns:
        if is_numeric(data, col):
            stats = data.groupby(class_att)[col].agg(['mean', 'std'])
            stats['std'] = stats['std'].replace(0, 1e-9)
            model[col] = ('gaussian', stats)
        else:
            mat = pd.crosstab(data[col], data[class_att])
            n_values = mat.shape[0]
            mat = (mat + smoothing) / (mat.sum(axis=0) + smoothing * n_values)
            model[col] = ('categorical', np.log(mat))

    return model


In [ ]:
def predict(model, instance):
    log_probs = {}

    for cls in model['_classes']:
        log_p = model['_apriori'][cls]

        for col, value in instance.items():
            if col not in model:
                continue

            attr_type, attr_data = model[col]

            if attr_type == 'gaussian':
                mean = attr_data.loc[cls, 'mean']
                std  = attr_data.loc[cls, 'std']
                log_p += -0.5 * np.log(2 * np.pi * std**2) - (value - mean)**2 / (2 * std**2)
            else:
                if value in attr_data.index:
                    log_p += attr_data.loc[value, cls]
                else:
                    log_p += np.log(1e-9)

        log_probs[cls] = log_p

    prediction = max(log_probs, key=log_probs.get)
    return prediction, log_probs


In [ ]:
def predict_batch(model, data):
    results = data.copy()
    predictions = []
    log_prob_cols = {cls: [] for cls in model['_classes']}

    for i in range(len(data)):
        pred, log_probs = predict(model, data.iloc[i])
        predictions.append(pred)
        for cls in model['_classes']:
            log_prob_cols[cls].append(log_probs[cls])

    results['prediction'] = predictions
    for cls in model['_classes']:
        results[f'log_P(class={cls})'] = log_prob_cols[cls]

    return results


In [ ]:
model = learn(data, class_att, smoothing=1.0)
model

In [ ]:
results = predict_batch(model, data.drop(columns=[class_att]))
results

In [ ]:
correct = (results['prediction'] == data[class_att]).sum()
f"Tacnost: {correct / len(data) * 100:.2f}%"